# 🚀 Full 142 GB Dataset Extraction - Spectral & Prosodic (Hybrid CPU/GPU)

This notebook automatically detects your hardware. 
- If a **T4 GPU** is attached, it will use ultra-fast PyTorch batching to extract features simultaneously.
- If **CPU only**, it will gracefully fall back to `librosa` multiprocessing.

**⚠️ PREREQUISITE:** You must manually upload `spectral_feature_extractor.py` and `prosodic_feature_extractor.py` to the Colab `/content` directory before running this!

In [ ]:
# 1. Setup Environment (Manual Upload Mode)
import os
import sys

if '/content' not in sys.path:
    sys.path.append('/content')
    
if not os.path.exists('/content/spectral_feature_extractor.py') or not os.path.exists('/content/prosodic_feature_extractor.py'):
    print("\n\u26d4 ERROR: Missing Python files!\n")
    print("Please drag and drop 'spectral_feature_extractor.py' and 'prosodic_feature_extractor.py' into the Colab file explorer on the left.")
    raise FileNotFoundError("Missing feature extractor files.")
else:
    print("\u2705 Feature extractors found successfully!")

!pip install -q zenodo_get pandas numpy librosa tqdm soundfile torchaudio

In [ ]:
# 2. Mount Drive & Setup Paths
from google.colab import drive
import shutil, tarfile, json, requests, torch
import pandas as pd, numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import concurrent.futures
import multiprocessing

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/142_Feature_Extracted')
SPECTRAL_DIR = BASE_DIR / 'spectral' / 'Dataset'
PROSODIC_DIR = BASE_DIR / 'prosodic' / 'Dataset'
SPECTRAL_DIR.mkdir(parents=True, exist_ok=True)
PROSODIC_DIR.mkdir(parents=True, exist_ok=True)

TEMP_DIR = Path('/content/temp_workspace')
TEMP_FLACS = TEMP_DIR / 'flacs'
TEMP_DIR.mkdir(parents=True, exist_ok=True)

STATE_FILE = BASE_DIR / 'processed_batches_cpu.json'
def load_state():
    if STATE_FILE.exists():
        with open(STATE_FILE, 'r') as f: return set(json.load(f))
    return set()

def save_state(s):
    with open(STATE_FILE, 'w') as f: json.dump(list(s), f)

processed_batches = load_state()
print(f"Workspace Ready. Processed batches: {len(processed_batches)}")

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\n⚡ HARDWARE DETECTED: {DEVICE.upper()} ⚡")

In [ ]:
# 3. Fetch Zenodo Info & Load Protocols
ZENODO_RECORD = '14498691'
API_URL = f"https://zenodo.org/api/records/{ZENODO_RECORD}"
res = requests.get(API_URL).json()
files_info = {f['key']: f['links']['self'] for f in res['files']}

PROTOCOL_TAR = 'ASVspoof5_protocols.tar.gz'
PROTOCOL_DIR = BASE_DIR / 'protocols'

if not PROTOCOL_DIR.exists():
    print("Downloading protocols from Zenodo...")
    PROTOCOL_DIR.mkdir(parents=True, exist_ok=True)
    !zenodo_get -r {ZENODO_RECORD} -g {PROTOCOL_TAR} -o {TEMP_DIR}
    with tarfile.open(TEMP_DIR / PROTOCOL_TAR, 'r:gz') as tar: tar.extractall(path=PROTOCOL_DIR)
    (TEMP_DIR / PROTOCOL_TAR).unlink()

protocol_map = {}
protocol_files = [f for f in PROTOCOL_DIR.rglob('*') if f.suffix in ['.txt', '.tsv']]

for split_file in protocol_files:
    with open(split_file, 'r') as f:
        for line in f:
            if line.startswith('#') or line.startswith('speaker_id') or not line.strip(): 
                continue
            parts = line.strip().split()
            if len(parts) >= 5:
                label = 0 if parts[4].lower() == 'bonafide' else 1
                fname = parts[1] if parts[1].endswith('.flac') else parts[1] + '.flac'
                protocol_map[fname] = label

print(f"Loaded {len(protocol_map)} entries into protocol_map.")

In [ ]:
# 4. Define Execution Branches (CPU Multiprocessing vs GPU Batching)

def process_audio_cpu(file_path):
    """Fallback: Process a single file using CPU (librosa)."""
    filename = file_path.name
    if filename not in protocol_map: return None
    try:
        import sys
        if '/content' not in sys.path:
            sys.path.append('/content')
            
        import librosa
        from spectral_feature_extractor import extract_spectral_row
        from prosodic_feature_extractor import extract_prosodic_row
        
        y, sr = librosa.load(str(file_path), sr=16000, mono=True)
        s_feat = extract_spectral_row(y, sr)
        p_feat = extract_prosodic_row(y, sr)
        
        s_feat['filename'] = p_feat['filename'] = filename
        s_feat['label'] = p_feat['label'] = protocol_map[filename]
        return s_feat, p_feat
    except Exception as e:
        return {'error': str(e), 'filename': filename}

def process_audio_gpu_batch(batch_paths):
    """Fast Path: Process up to 64 files at once using PyTorch on GPU."""
    import torchaudio
    from spectral_feature_extractor import extract_spectral_batch
    from prosodic_feature_extractor import extract_prosodic_batch
    
    valid_paths = [p for p in batch_paths if p.name in protocol_map]
    if not valid_paths: return [], []
    
    # Load and pad waveforms to exactly 5 seconds (80,000 samples at 16k)
    target_samples = 16000 * 5
    waveforms = []
    for p in valid_paths:
        w, sr = torchaudio.load(str(p))
        w = w.mean(dim=0) # mono
        if w.size(0) > target_samples:
            w = w[:target_samples]
        elif w.size(0) < target_samples:
            w = torch.nn.functional.pad(w, (0, target_samples - w.size(0)))
        waveforms.append(w)
        
    wave_tensor = torch.stack(waveforms).to('cuda') # (batch, time)
    
    # GPU Batch Extraction
    s_batch_res = extract_spectral_batch(wave_tensor, sr=16000, device='cuda')
    p_batch_res = extract_prosodic_batch(wave_tensor, sr=16000, device='cuda')
    
    for i, path in enumerate(valid_paths):
        s_batch_res[i]['filename'] = p_batch_res[i]['filename'] = path.name
        s_batch_res[i]['label'] = p_batch_res[i]['label'] = protocol_map[path.name]
        
    return s_batch_res, p_batch_res


In [ ]:
# 5. Master Extraction Loop
tar_files = [f for f in files_info.keys() if f.startswith('flac_') and f.endswith('.tar')]

for tar_name in sorted(tar_files):
    if tar_name in processed_batches: 
        continue
    
    print(f"\n{'='*50}\nProcessing {tar_name}...\n{'='*50}")
    url = files_info[tar_name]
    tar_path = TEMP_DIR / tar_name
    !wget -q -O {tar_path} {url}
    
    if TEMP_FLACS.exists(): shutil.rmtree(TEMP_FLACS)
    TEMP_FLACS.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, 'r') as tar: 
        if hasattr(tarfile, 'data_filter'):
            tar.extractall(path=TEMP_FLACS, filter='data')
        else:
            tar.extractall(path=TEMP_FLACS)
    
    all_flacs = list(TEMP_FLACS.rglob('*.flac'))
    print(f"Found {len(all_flacs)} audio files in {tar_name}.")
    
    split_char = tar_name.split('_')[1]
    split_name = 'train' if split_char == 'T' else 'dev' if split_char == 'D' else 'eval'
    s_csv = SPECTRAL_DIR / f"spectral_{split_name}.csv"
    p_csv = PROSODIC_DIR / f"prosodic_{split_name}.csv"
    
    s_results, p_results, errors = [], [], []
    
    if DEVICE == 'cuda':
        print("Running GPU Batch Extraction (Batch Size = 64)...")
        batch_size = 64
        for i in tqdm(range(0, len(all_flacs), batch_size)):
            batch_paths = all_flacs[i:i+batch_size]
            try:
                s_batch, p_batch = process_audio_gpu_batch(batch_paths)
                s_results.extend(s_batch)
                p_results.extend(p_batch)
            except Exception as e:
                errors.append({'error': str(e), 'batch_start': i})
    else:
        workers = multiprocessing.cpu_count()
        print(f"Running CPU Extraction on {workers} cores...")
        with concurrent.futures.ProcessPoolExecutor(max_workers=workers) as executor:
            for res in tqdm(executor.map(process_audio_cpu, all_flacs, chunksize=100), total=len(all_flacs)):
                if res is None:
                    continue
                if isinstance(res, dict) and 'error' in res:
                    errors.append(res)
                else:
                    s_results.append(res[0])
                    p_results.append(res[1])
                
    if errors:
        print(f"\n⚠️ {len(errors)} files/batches failed. Sample error: {errors[0]}")
        
    if s_results:
        print(f"Saving {len(s_results)} rows to {s_csv.name} & {p_csv.name}...")
        pd.DataFrame(s_results).to_csv(s_csv, mode='a', header=not s_csv.exists(), index=False)
        pd.DataFrame(p_results).to_csv(p_csv, mode='a', header=not p_csv.exists(), index=False)
        
    shutil.rmtree(TEMP_FLACS)
    tar_path.unlink()
    processed_batches.add(tar_name)
    save_state(processed_batches)

print("🎉 Dataset Extraction Complete!")